<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Copy_of_notebooks_01_real_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pandas==2.2.3", "pyarrow==21.0.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', 'pandas==2.2.3', 'pyarrow==21.0.0'], returncode=0)

In [12]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import ast, csv, hashlib, importlib.metadata, json, os, platform, re, shutil, subprocess, sys, tempfile
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path('/content/ichi2027_neuroimaging_phase01') if Path('/content').is_dir() else Path.cwd()/'ichi2027_neuroimaging_phase01'
REPOS = ROOT / 'upstream'
INPUT = ROOT / 'authorized_input'
OUTPUT = ROOT / 'shareable_output'
PRIVATE = ROOT / 'private_manifests'
for folder in [REPOS, INPUT, OUTPUT, PRIVATE]:
    folder.mkdir(parents=True, exist_ok=True)

PROJECT = 'Neuroimaging FHIR-to-OMOP ICHI 2027'
REPO_URLS = {
    'ohdsi_tfl': 'https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',
    'neurofhir_qc': 'https://github.com/SANGHATI23/neurofhir-qc.git',
    'imagewg': 'https://github.com/OHDSI/ImageWG.git',
    'pyomop': 'https://github.com/dermatologist/pyomop.git',
}
FREEZE_TAG = 'tfl-submission-freeze-v15'
RUN_UTC = datetime.now(timezone.utc).isoformat()

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def command(args, timeout=240, cwd=None):
    try:
        p=subprocess.run(args,cwd=cwd,capture_output=True,text=True,timeout=timeout)
        return {'ok':p.returncode==0,'code':p.returncode,'stdout':p.stdout.strip()[-2000:],'stderr':p.stderr.strip()[-500:]}
    except Exception as exc:
        return {'ok':False,'code':None,'stdout':'','stderr':type(exc).__name__+': '+str(exc)[:350]}

report = {
    'project':PROJECT, 'phase':'01 - upstream evidence + authorized data inventory',
    'run_utc':RUN_UTC, 'scientific_result_status':'NOT_RUN',
    'comparative_arm_metrics':None, 'adni_cohort_metrics':None,
    'environment':{'python':sys.version.split()[0],'platform':platform.platform(),
                   'pandas':pd.__version__,'pyarrow':importlib.metadata.version('pyarrow')},
    'repositories':{},'schema':{},'source_inventory':[], 'gates':{}
}
print(json.dumps({'phase':report['phase'],'run_utc':RUN_UTC,'python':report['environment']['python']},indent=2))

{
  "phase": "01 - upstream evidence + authorized data inventory",
  "run_utc": "2026-09-26T21:31:36.470545+00:00",
  "python": "3.13.15"
}


In [13]:
for name,url in REPO_URLS.items():
    dest=REPOS/name
    rec={'url':url,'status':'NOT_TESTED','commit':None,'freeze_reference':None,'errors':[]}
    if not (dest/'.git').is_dir():
        clone=command(['git','clone','--quiet','--depth','1',url,str(dest)],timeout=240)
        if not clone['ok']:
            rec.update(status='UNAVAILABLE',errors=[clone['stderr']])
            report['repositories'][name]=rec
            print(name,rec['status'],rec['errors'][0]); continue
    elif command(['git','-C',str(dest),'status','--porcelain'])['stdout']:
        rec.update(status='BLOCKED_LOCAL_CHANGES',errors=['Existing local checkout is modified; refusing to overwrite.'])
        report['repositories'][name]=rec
        print(name,rec['status']); continue
    else:
        fetch=command(['git','-C',str(dest),'fetch','--quiet','--depth','1','origin'],timeout=180)
        if not fetch['ok']:
            rec['errors'].append('fetch: '+fetch['stderr'])
    if name=='ohdsi_tfl':
        tag_exists=command(['git','ls-remote','--tags','--refs',url,'refs/tags/'+FREEZE_TAG],timeout=50)
        if tag_exists['ok'] and tag_exists['stdout']:
            got=command(['git','-C',str(dest),'fetch','--quiet','--depth','1','origin','refs/tags/'+FREEZE_TAG],timeout=180)
            checked=command(['git','-C',str(dest),'checkout','--quiet','--detach','FETCH_HEAD']) if got['ok'] else got
            if checked['ok']:
                rec['freeze_reference']=FREEZE_TAG
            else:
                rec['errors'].append('freeze checkout failed: '+checked['stderr'])
        else:
            rec['errors'].append('Exact v15 git tag not found or remote tag query inaccessible; HEAD is NOT a verified v15 freeze.')
    head=command(['git','-C',str(dest),'rev-parse','HEAD'])
    if head['ok'] and re.fullmatch(r'[0-9a-f]{40}',head['stdout']):
        rec['commit']=head['stdout'];rec['status']='PINNED' if not rec['errors'] else 'PINNED_WITH_WARNINGS'
        command(['git','-C',str(dest),'checkout','--quiet','--detach',rec['commit']])
    else:
        rec.update(status='UNAVAILABLE',errors=rec['errors']+[head['stderr']])
    report['repositories'][name]=rec
    print(f"{name}: {rec['status']} | commit={rec['commit']} | exact_v15_tag={bool(rec['freeze_reference'])}")

report['gates']['all_repositories_accessible']=all(x['commit'] for x in report['repositories'].values())
report['gates']['exact_original_tfl_v15_tag_verified']=bool(report['repositories']['ohdsi_tfl']['freeze_reference'])

ohdsi_tfl: PINNED | commit=a4b7265adabf096e84b957de068e24a5ab853425 | exact_v15_tag=True
neurofhir_qc: PINNED | commit=88c1a6b07420b62719ef9efcec7494a86446fa39 | exact_v15_tag=False
imagewg: PINNED | commit=8f0a19bb94e941bb335a097bc9851fab672621c7 | exact_v15_tag=False
pyomop: PINNED | commit=dfaa6d4062dbc600f52f5377019a347c4caa2cc7 | exact_v15_tag=False


In [14]:
TERMS=('fhir','omop','tfl','fidelity','trace','provenance','image','segment','measurement','etl','map','convert','transform')
def code_inventory(root):
    if not root.is_dir(): return {'status':'UNAVAILABLE'}
    pyfiles=[p for p in root.rglob('*.py') if not any(x in {'.git','.venv','venv','site-packages','build','dist'} for x in p.parts)]
    allfiles=[p for p in root.rglob('*') if p.is_file() and '.git' not in p.parts]
    findings=[]; syntax_errors=[]
    for path in sorted(pyfiles)[:1200]:
        rel=str(path.relative_to(root))
        try:
            src=path.read_text(encoding='utf-8',errors='replace')
            tree=ast.parse(src,filename=rel)
            hits=[{'symbol':node.name,'line':node.lineno} for node in ast.walk(tree)
                  if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))
                  and any(term in node.name.lower() for term in TERMS)]
            if hits or any(t in rel.lower() for t in TERMS):
                findings.append({'file':rel,'sha256':hashlib.sha256(src.encode()).hexdigest(),'matched_symbols':hits[:45]})
        except SyntaxError as exc: syntax_errors.append({'file':rel,'line':exc.lineno,'message':exc.msg})
    manifests=[str(p.relative_to(root)) for p in allfiles if p.name in {'pyproject.toml','setup.py','requirements.txt','requirements-dev.txt','environment.yml','Dockerfile','README.md'}]
    return {'status':'INSPECTED','total_python_files':len(pyfiles),'total_repo_files':len(allfiles),
            'manifests':sorted(manifests)[:80], 'implementation_candidates':findings[:100],
            'python_parse_errors':syntax_errors[:35]}

for name in REPO_URLS:
    rec=report['repositories'][name]
    rec['static_inventory']=code_inventory(REPOS/name) if rec['commit'] else {'status':'UNAVAILABLE'}
    scan=rec['static_inventory']
    print(name,'python_files=',scan.get('total_python_files'),'candidate_paths=',len(scan.get('implementation_candidates',[])),
          'syntax_errors=',len(scan.get('python_parse_errors',[])))

ohdsi_tfl python_files= 11 candidate_paths= 3 syntax_errors= 0
neurofhir_qc python_files= 2 candidate_paths= 0 syntax_errors= 0
imagewg python_files= 0 candidate_paths= 0 syntax_errors= 0
pyomop python_files= 45 candidate_paths= 27 syntax_errors= 0


In [15]:
imaging=REPOS/'imagewg'
if report['repositories']['imagewg']['commit']:
    schema_candidates=[]
    for p in sorted(imaging.rglob('*')):
        if not p.is_file() or '.git' in p.parts or p.suffix.lower() not in ('.sql','.csv','.json','.yaml','.yml','.md','.txt','.tsv','.xlsx'):
            continue
        if p.stat().st_size > 2_000_000: continue
        path=str(p.relative_to(imaging))
        if p.suffix.lower()=='.xlsx':
            if 'schema' in path.lower() or 'cdm' in path.lower():
                schema_candidates.append({'file':path,'sha256':sha256(p),'kind':'xlsx_not_parsed','create_tables':[]})
            continue
        try: contents=p.read_text(encoding='utf-8',errors='replace')
        except Exception: continue
        mentions=bool(re.search(r'image[_ ]occurrence|image[_ ]feature',contents,re.I))
        if mentions or ('image' in path.lower() and ('schema' in path.lower() or p.suffix=='.sql')):
            tables=sorted({x.lower() for x in re.findall(r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?["`\[]?([A-Za-z_][\w.]*)',contents,re.I)})
            schema_candidates.append({'file':path,'sha256':sha256(p),'kind':p.suffix.lower(),
                                      'create_tables':tables[:50],
                                      'mentions_image_occurrence':bool(re.search(r'image[_ ]occurrence',contents,re.I)),
                                      'mentions_image_feature':bool(re.search(r'image[_ ]feature',contents,re.I))})
    report['schema']={'imagewg_commit':report['repositories']['imagewg']['commit'],
                      'candidate_files':schema_candidates[:100],
                      'ddl_with_both_tables': [
                          x['file'] for x in schema_candidates
                          if x['kind']=='.sql' and any('image_occurrence' in t for t in x['create_tables'])
                          and any('image_feature' in t for t in x['create_tables'])],
                      'ddl_image_occurrence': [x['file'] for x in schema_candidates if x['kind']=='.sql' and any('image_occurrence' in t for t in x['create_tables'])],
                      'ddl_image_feature': [x['file'] for x in schema_candidates if x['kind']=='.sql' and any('image_feature' in t for t in x['create_tables'])]}
else: report['schema']={'status':'IMAGEWG_REPOSITORY_UNAVAILABLE','candidate_files':[],'ddl_with_both_tables':[],'ddl_image_occurrence':[],'ddl_image_feature':[]}
report['gates']['schema_ddl_discovered']=bool(report['schema']['ddl_image_occurrence'] and report['schema']['ddl_image_feature'])
print('ImageWG revision:',report['schema'].get('imagewg_commit','unavailable'))
print('MI-CDM DDL image_occurrence:',report['schema']['ddl_image_occurrence'])
print('MI-CDM DDL image_feature:',report['schema']['ddl_image_feature'])

ImageWG revision: 8f0a19bb94e941bb335a097bc9851fab672621c7
MI-CDM DDL image_occurrence: []
MI-CDM DDL image_feature: []


In [16]:
AUTHORIZED_FORMATS={'.csv','.tsv','.parquet','.ndjson','.json','.zip'}
COL_GROUPS={
    'participant_id':('participant_id','person_id','patient_id','rid','ptid','subject_id'),
    'scan_or_study_id':('scan_id','study_id','imageuid','image_uid','studyinstanceuid','study_instance_uid'),
    'series_or_acquisition_id':('series_id','seriesuid','series_instance_uid','acquisition_id'),
    'acquisition_date':('acquisition_date','scan_date','examdate','study_date','image_date'),
    'timepoint':('timepoint','visit','viscode','viscode2','visit_code','session'),
    'anatomical_region':('region','anatomical_region','structure','brain_region'),
    'laterality':('laterality','hemisphere','side'),
    'volume_value':('value','volume','volume_mm3','volume_cm3','measurement_value'),
    'volume_unit':('unit','volume_unit','unit_code','ucum_unit'),
    'baseline_diagnosis':('diagnosis_at_baseline','baseline_diagnosis','dx_bl','dxbaseline'),
    'extraction_model':('model_name','software_name','segmentation_software','extractor'),
    'model_version':('model_version','software_version','freesurfer_version'),
    'qc':('qc','qc_state','quality_control','qc_status'),
    'trace_state':('trace_state','reliability_state'),
    'trace_rule_version':('trace_rule_version','reliability_rule_version'),
    'trace_reference':('trace_reference_id','trace_reference','calibration_reference_id'),
}
def norm(s):return re.sub('[^a-z0-9]','',s.lower())
def detect_columns(columns):
    index={}
    for c in columns:index.setdefault(norm(str(c)),[]).append(str(c))
    result={}
    for group,aliases in COL_GROUPS.items():
        matches=sorted({name for a in aliases for name in index.get(norm(a),[])})
        result[group]={'matches':matches,'status':'MISSING' if not matches else 'EXACTLY_ONE' if len(matches)==1 else 'AMBIGUOUS'}
    return result

def audit_table(file):
    if file.suffix.lower()=='.parquet':
        table=pq.ParquetFile(file)
        cols=table.schema_arrow.names
        rows=table.metadata.num_rows
        return {'rows':int(rows),'columns':cols,'column_groups':detect_columns(cols),'format':'parquet'}
    sep='\t' if file.suffix.lower()=='.tsv' else ','
    rows=0; columns=None; failed=None
    try:
        for chunk in pd.read_csv(file,sep=sep,chunksize=50000,dtype=str,encoding='utf-8',on_bad_lines='error'):
            if columns is None:columns=list(chunk.columns)
            rows+=len(chunk)
        if columns is None:columns=list(pd.read_csv(file,sep=sep,nrows=0).columns)
    except Exception as exc:
        failed=type(exc).__name__+': '+str(exc)[:180]
    return {'rows':rows if failed is None else None,'columns':columns or [],
            'column_groups':detect_columns(columns or []),'format':file.suffix[1:],'read_error':failed}

def audit_fhir(file):
    counts=Counter(); n=0; invalid=0
    if file.suffix.lower()=='.ndjson':
        with file.open(encoding='utf-8') as handle:
            for line in handle:
                if not line.strip():continue
                n+=1
                try:counts[json.loads(line).get('resourceType','MISSING')]+=1
                except (ValueError,AttributeError):invalid+=1
    else:
        with file.open(encoding='utf-8') as handle:
            obj=json.load(handle)
        if isinstance(obj,dict) and obj.get('resourceType')=='Bundle':
            for x in obj.get('entry',[]): counts[x.get('resource',{}).get('resourceType','MISSING')]+=1;n+=1
        elif isinstance(obj,dict): counts[obj.get('resourceType','UNKNOWN')]+=1;n=1
        else:raise ValueError('JSON is not a FHIR resource or Bundle')
    return {'format':'FHIR','total_resources':n,'resource_types':dict(counts),'invalid_ndjson_lines':invalid}

In [17]:

from pathlib import Path
import shutil

files = sorted(Path("/content").glob("TRACE_ADNI*.csv"))

INPUT.mkdir(parents=True, exist_ok=True)

for file in files:
    shutil.copy2(file, INPUT / file.name)
    print("Ready:", file.name)

print("Total files:", len(files))

Ready: TRACE_ADNI1_landmark12_REAL.csv
Ready: TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv
Ready: TRACE_ADNI1_repeat_pairs_REAL.csv
Total files: 3


In [18]:
drive_dir=Path('/content/drive/MyDrive/NeuroFHIR_ICHI2027/inputs')
preexisting=[p for p in INPUT.iterdir() if p.is_file() and p.suffix.lower() in AUTHORIZED_FORMATS]
if not preexisting and not drive_dir.exists():
    try:
        from google.colab import files
        print('Choose an AUTHORIZED export ONLY if your data-use agreement permits processing in Google Colab. Cancel to audit repositories without private data.')
        chosen=files.upload()
        for filename,data in chosen.items():
            safe=Path(filename).name
            if safe.startswith('.') or Path(safe).suffix.lower() not in AUTHORIZED_FORMATS:
                continue
            (INPUT/safe).write_bytes(data)
        del chosen
    except ImportError:
        print('No Colab upload interface in this environment. Any existing files in',INPUT,'will be audited.')
    except Exception as exc:
        print('No authorized upload provided:',type(exc).__name__)

source_dir=drive_dir if drive_dir.exists() else INPUT
files_to_audit=sorted(p for p in source_dir.rglob('*') if p.is_file() and p.suffix.lower() in AUTHORIZED_FORMATS)
report['gates']['authorized_input_available']=bool(files_to_audit)
print('Input files found:',len(files_to_audit))

Input files found: 3


In [19]:
MAX_FILES=100
private_fingerprints=[]
for number,path in enumerate(files_to_audit[:MAX_FILES],1):
    item={'source_key':f'source_{number:03d}','format':path.suffix.lower(),'status':'NOT_READ',
          'bytes':path.stat().st_size}
    private_fingerprints.append({'source_key':item['source_key'],'local_path':str(path),'sha256':sha256(path)})
    try:
        if path.suffix.lower() in ('.csv','.tsv','.parquet'):
            item.update(audit_table(path));item['status']='AUDITED' if not item.get('read_error') else 'READ_ERROR'
        elif path.suffix.lower() in ('.json','.ndjson'):
            item.update(audit_fhir(path));item['status']='AUDITED'
        else:
            import zipfile
            with zipfile.ZipFile(path) as z:
                allowed=[x for x in z.infolist() if not x.is_dir() and Path(x.filename).suffix.lower() in AUTHORIZED_FORMATS-{'.zip'}]
                item.update({'format':'zip','contained_eligible_files':len(allowed),'contained_formats':dict(Counter(Path(x.filename).suffix.lower() for x in allowed))})
            item['status']='ARCHIVE_INVENTORIED_NOT_EXTRACTED'
    except Exception as exc:
        item.update(status='READ_ERROR',error=type(exc).__name__+': '+str(exc)[:180])
    report['source_inventory'].append(item)
    if item.get('column_groups'):
        missing=[k for k,v in item['column_groups'].items() if v['status']!='EXACTLY_ONE']
        print(item['source_key'],item['format'],'rows=',item.get('rows'),'unresolved_field_groups=',missing)
    else:print(item['source_key'],item['format'],item['status'])

(PRIVATE/'local_source_fingerprints_DO_NOT_SHARE.json').write_text(json.dumps(private_fingerprints,indent=2))
if len(files_to_audit)>MAX_FILES:
    report['source_file_limit_warning']=f'{len(files_to_audit)} supported files exist; first {MAX_FILES} inventoried'
report['gates']['source_field_contract_covered_in_single_table']=any(
    all(x.get('column_groups',{}).get(k,{}).get('status')=='EXACTLY_ONE' for k in
        ['participant_id','scan_or_study_id','acquisition_date','timepoint','anatomical_region',
         'laterality','volume_value','volume_unit','baseline_diagnosis','extraction_model','model_version'])
    for x in report['source_inventory'] if x['status']=='AUDITED' and (x.get('rows') or 0)>0
)
report['gates']['original_trace_available_in_single_table']=any(
    all(x.get('column_groups',{}).get(k,{}).get('status')=='EXACTLY_ONE' for k in
        ['trace_state','trace_rule_version','trace_reference'])
    for x in report['source_inventory'] if x['status']=='AUDITED' and (x.get('rows') or 0)>0
)
report['gates']['actual_arm_adapters_implemented_and_tested']=False
report['gates']['omop_vocabulary_snapshot_pinned']=False
report['gates']['source_cohort_query_approved']=False
report['gates']['ready_for_formal_comparison']=False
print('Evidence gates:',json.dumps(report['gates'],indent=2))

source_001 csv rows= 199 unresolved_field_groups= ['participant_id', 'scan_or_study_id', 'series_or_acquisition_id', 'acquisition_date', 'timepoint', 'anatomical_region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference']
source_002 csv rows= 258 unresolved_field_groups= ['participant_id', 'scan_or_study_id', 'series_or_acquisition_id', 'acquisition_date', 'timepoint', 'anatomical_region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference']
source_003 csv rows= 90 unresolved_field_groups= ['scan_or_study_id', 'series_or_acquisition_id', 'anatomical_region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference']
Evidence gates: {
  "all_repositories_acc

In [20]:
report['next_stage_requirements']=[
    'Review actual OHDSI v15 entrypoints and original implementation tests',
    'Pin and inspect exact MI-CDM schema DDL and OMOP vocabulary snapshot',
    'Reconcile authorized source table layout and identifiers without inferred metadata',
    'Prespecify source-defined MCI baseline/12-month bilateral hippocampus eligibility',
    'Execute genuine four-arm ETL comparison only after audited adapters exist',
]
report['prohibited_interpretation']='Repository inventory and schema/field coverage are not ETL efficacy findings.'
report['scientific_result_status']='NOT_RUN_PENDING_REAL_ETL_AND_VERIFIED_INPUT'
out=OUTPUT/'ICHI2027_Phase01_Audit_SHAREABLE.json'
out.write_text(json.dumps(report,indent=2,ensure_ascii=False))
print('PHASE 01 COMPLETE. NO SCIENTIFIC RESULT WAS GENERATED.')
print('Shareable audit:',out.name)
print(json.dumps({'phase':report['phase'],'repositories':{
    k:{'status':v['status'],'commit':v['commit'],'exact_v15_tag':bool(v['freeze_reference'])}
    for k,v in report['repositories'].items()},'gates':report['gates'],
    'input_summary':[{'source_key':x['source_key'],'status':x['status'],'format':x['format'],
                      'rows':x.get('rows'),'total_resources':x.get('total_resources')}
                     for x in report['source_inventory']]},indent=2))
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('Audit JSON stored at',out)
except Exception as exc:
    print('Automatic download unavailable; audit JSON stored at',out, type(exc).__name__)

PHASE 01 COMPLETE. NO SCIENTIFIC RESULT WAS GENERATED.
Shareable audit: ICHI2027_Phase01_Audit_SHAREABLE.json
{
  "phase": "01 - upstream evidence + authorized data inventory",
  "repositories": {
    "ohdsi_tfl": {
      "status": "PINNED",
      "commit": "a4b7265adabf096e84b957de068e24a5ab853425",
      "exact_v15_tag": true
    },
    "neurofhir_qc": {
      "status": "PINNED",
      "commit": "88c1a6b07420b62719ef9efcec7494a86446fa39",
      "exact_v15_tag": false
    },
    "imagewg": {
      "status": "PINNED",
      "commit": "8f0a19bb94e941bb335a097bc9851fab672621c7",
      "exact_v15_tag": false
    },
    "pyomop": {
      "status": "PINNED",
      "commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7",
      "exact_v15_tag": false
    }
  },
  "gates": {
    "all_repositories_accessible": true,
    "exact_original_tfl_v15_tag_verified": true,
    "schema_ddl_discovered": false,
    "authorized_input_available": true,
    "source_field_contract_covered_in_single_table": false,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>